In [1]:
from pathlib import Path
import subprocess
import sys

# 1. Locate the Rust project root relative to current working directory
project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "Cargo.toml").exists()),
    None,
)

if project_root is None:
    print("Build Failed: Could not locate Cargo.toml in the current path or parent directories.")
else:
    wheel_dir = project_root / "target" / "wheels"
    wheel_dir.mkdir(parents=True, exist_ok=True)

    try:
        # 2. Build the wheel package quietly
        subprocess.run(
            [
                sys.executable,
                "-m",
                "maturin",
                "build",
                "--manifest-path",
                str(project_root / "Cargo.toml"),
                "--out",
                str(wheel_dir),
            ],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
            text=True,
        )

        # 3. Locate the generated wheel
        wheels = sorted(wheel_dir.glob("fraud_spike_detector-*.whl"))
        if not wheels:
            print("Build Failed: Wheel file was not generated.")
        else:
            # 4. Install the wheel into the current active Python interpreter
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", str(wheels[-1])],
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.PIPE,
                text=True,
            )
            print("Build & Installation Successful! 'fraud_spike_detector' is ready to use.")

    except subprocess.CalledProcessError as e:
        print("Build Failed during execution.")
        if e.stderr:
            print(f"Error Details:\n{e.stderr.strip()}")

Build & Installation Successful! 'fraud_spike_detector' is ready to use.


In [2]:
try:
    import fraud_spike_detector

    # Initialize the CUSUM streaming detector
    streaming_layer = fraud_spike_detector.PyStreamingLayer(
        alpha=0.05,
        cusum_threshold=4.0,
        cusum_drift=0.5
    )

    # Construct sample transactions
    sample_transactions = [
        fraud_spike_detector.PyTransaction(
            id="tx_001",
            merchant_id="merchant_123",
            bin="411111",
            is_disputed=False,
            timestamp_ms=1725249600000
        ),
        fraud_spike_detector.PyTransaction(
            id="tx_002",
            merchant_id="merchant_123",
            bin="411111",
            is_disputed=True,
            timestamp_ms=1725249601000
        ),
    ]

    # Process batch
    alerts = streaming_layer.process_transaction_batch(sample_transactions)

    print("Rust Streaming Engine loaded and running successfully!")
    print(f"Processed: {len(sample_transactions)} transactions")
    print(f"Alerts detected: {len(alerts)}")
except ImportError:
    print("Rust module not found. Continuing with ML pipeline execution.")

Rust Streaming Engine loaded and running successfully!
Processed: 2 transactions
Alerts detected: 0


In [3]:
import time
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)

# =====================================================================
# 1. DATASET GENERATION
# =====================================================================

def generate_fraud_dataset(
    n_samples: int = 50_000, fraud_rate: float = 0.02, seed: int = 42
):
    rng = np.random.default_rng(seed)
    n_fraud = int(n_samples * fraud_rate)
    n_clean = n_samples - n_fraud

    # Stage 1 Features
    amount_clean = rng.lognormal(mean=3.5, sigma=1.0, size=n_clean)
    velocity_clean = rng.poisson(lam=1.2, size=n_clean)
    hour_clean = rng.integers(0, 24, size=n_clean)
    risk_cat_clean = rng.choice([0, 1, 2], p=[0.7, 0.2, 0.1], size=n_clean)

    amount_fraud = rng.lognormal(mean=4.2, sigma=1.2, size=n_fraud)
    velocity_fraud = rng.poisson(lam=3.8, size=n_fraud)
    hour_fraud = rng.choice([0, 1, 2, 3, 4, 22, 23], size=n_fraud)
    risk_cat_fraud = rng.choice([0, 1, 2], p=[0.1, 0.3, 0.6], size=n_fraud)

    # Stage 2 Features
    device_shared_clean = rng.binomial(n=1, p=0.05, size=n_clean)
    ip_reputation_clean = rng.beta(a=8, b=2, size=n_clean)
    merchant_cb_rate_clean = rng.exponential(scale=0.005, size=n_clean)
    graph_risk_score_clean = rng.beta(a=2, b=8, size=n_clean)

    device_shared_fraud = rng.binomial(n=1, p=0.45, size=n_fraud)
    ip_reputation_fraud = rng.beta(a=2, b=5, size=n_fraud)
    merchant_cb_rate_fraud = rng.exponential(scale=0.04, size=n_fraud)
    graph_risk_score_fraud = rng.beta(a=6, b=3, size=n_fraud)

    df_clean = pd.DataFrame({
        "amount": amount_clean,
        "velocity_1h": velocity_clean,
        "hour_of_day": hour_clean,
        "merchant_risk_cat": risk_cat_clean,
        "device_shared_count": device_shared_clean,
        "ip_reputation_score": ip_reputation_clean,
        "merchant_historical_cb_rate": merchant_cb_rate_clean,
        "graph_cluster_risk": graph_risk_score_clean,
        "is_fraud": 0,
    })

    df_fraud_data = pd.DataFrame({
        "amount": amount_fraud,
        "velocity_1h": velocity_fraud,
        "hour_of_day": hour_fraud,
        "merchant_risk_cat": risk_cat_fraud,
        "device_shared_count": device_shared_fraud,
        "ip_reputation_score": ip_reputation_fraud,
        "merchant_historical_cb_rate": merchant_cb_rate_fraud,
        "graph_cluster_risk": graph_risk_score_fraud,
        "is_fraud": 1,
    })

    df = pd.concat([df_clean, df_fraud_data]).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    df["merchant_risk_cat"] = df["merchant_risk_cat"].astype("category")
    df["timestamp"] = pd.date_range(start="2026-08-01", periods=len(df), freq="1s")
    return df

# =====================================================================
# 2. MODEL CLASSES
# =====================================================================

class Stage1HotPath:
    def __init__(self, target_recall: float = 0.99):
        self.target_recall = target_recall
        self.scaler = StandardScaler()
        self.model = LogisticRegression(penalty="l2", C=1.0, max_iter=500, class_weight="balanced")
        self.threshold = 0.5

    def fit(self, X: pd.DataFrame, y: pd.Series):
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
        
        X_train_scaled = self.scaler.fit_transform(X_train)
        self.model.fit(X_train_scaled, y_train)
        
        X_val_scaled = self.scaler.transform(X_val)
        probs_val = self.model.predict_proba(X_val_scaled)[:, 1]

        thresholds = np.linspace(0.001, 0.999, 1000)
        valid_thresholds = []

        for th in thresholds:
            preds = (probs_val >= th).astype(int)
            true_positives = np.sum((preds == 1) & (y_val == 1))
            total_positives = np.sum(y_val == 1)
            recall = true_positives / total_positives if total_positives > 0 else 1.0

            if recall >= self.target_recall:
                valid_thresholds.append(th)

        self.threshold = max(valid_thresholds) if valid_thresholds else 0.01

    def predict_escalate(self, X: pd.DataFrame):
        X_scaled = self.scaler.transform(X)
        start_t = time.perf_counter()
        probs = self.model.predict_proba(X_scaled)[:, 1]
        latencies_ms = (time.perf_counter() - start_t) * 1000 / len(X)
        escalate_mask = probs >= self.threshold
        return escalate_mask, probs, latencies_ms

class Stage2WarmPath:
    def __init__(self, cost_ratio: float = 10.0):
        self.cost_ratio = cost_ratio
        self.model = None

    def fit(self, X: pd.DataFrame, y: pd.Series):
        train_data = lgb.Dataset(X, label=y, categorical_feature=["merchant_risk_cat"])
        params = {
            "objective": "binary",
            "metric": "binary_logloss",
            "boosting_type": "gbdt",
            "scale_pos_weight": self.cost_ratio,
            "num_leaves": 31,
            "learning_rate": 0.05,
            "feature_fraction": 0.8,
            "verbose": -1,
            "seed": 42,
        }
        self.model = lgb.train(params, train_data, num_boost_round=150)

    def predict_proba(self, X: pd.DataFrame):
        start_t = time.perf_counter()
        probs = self.model.predict(X)
        latency_ms = (time.perf_counter() - start_t) * 1000 / max(len(X), 1)
        probs_2d = np.column_stack([1 - probs, probs])
        return probs_2d, latency_ms

class SplitConformalPredictor:
    def __init__(self, alpha: float = 0.05):
        self.alpha = alpha
        self.q_hat = None

    def calibrate(self, cal_probs: np.ndarray, cal_labels: np.ndarray):
        n = len(cal_labels)
        true_class_probs = cal_probs[np.arange(n), cal_labels]
        scores = 1.0 - true_class_probs
        quantile_val = np.ceil((n + 1) * (1 - self.alpha)) / n
        quantile_val = min(1.0, quantile_val)
        self.q_hat = np.quantile(scores, quantile_val, method="higher")

    def predict_set(self, test_probs: np.ndarray):
        if self.q_hat is None:
            raise ValueError("Predictor must be calibrated before inference.")

        s0 = 1.0 - test_probs[:, 0]
        s1 = 1.0 - test_probs[:, 1]

        in_set_0 = s0 <= self.q_hat
        in_set_1 = s1 <= self.q_hat

        pred_sets = []
        for p0_in, p1_in, p_vec in zip(in_set_0, in_set_1, test_probs):
            s = set()
            if p0_in: s.add(0)
            if p1_in: s.add(1)
            if not s:
                s.add(int(np.argmax(p_vec)))
            pred_sets.append(s)

        return pred_sets

    @staticmethod
    def evaluate_coverage(pred_sets, true_labels):
        covered = [y in p_set for y, p_set in zip(true_labels, pred_sets)]
        set_sizes = [len(s) for s in pred_sets]
        coverage = np.mean(covered)
        avg_set_size = np.mean(set_sizes)
        ambiguous_pct = np.mean([1 if len(s) > 1 else 0 for s in pred_sets])
        return coverage, avg_set_size, ambiguous_pct

# =====================================================================
# 3. VECTORIZED FINANCIAL EVALUATION
# =====================================================================

def calculate_net_saved_margin(test_df, pred_sets, esc_mask_test, cost_per_review=15.0, chargeback_fee=25.0):
    test_escalated = test_df[esc_mask_test].copy().reset_index(drop=True)
    actual_fraud = test_escalated["is_fraud"].values == 1
    amounts = test_escalated["amount"].values

    is_block = np.array([p_set == {1} for p_set in pred_sets])
    is_review = np.array([p_set == {0, 1} for p_set in pred_sets])
    is_pass = np.array([p_set == {0} for p_set in pred_sets])

    cleared_by_s1 = test_df[~esc_mask_test]
    s1_missed_fraud = cleared_by_s1[cleared_by_s1["is_fraud"] == 1]
    s1_missed_cost = s1_missed_fraud["amount"].sum() + (len(s1_missed_fraud) * chargeback_fee)

    fraud_prevented = np.sum(amounts[is_block & actual_fraud]) + np.sum(amounts[is_review & actual_fraud])
    review_costs = np.sum(is_review) * cost_per_review
    missed_s2_fraud = np.sum(amounts[is_pass & actual_fraud]) + (np.sum(is_pass & actual_fraud) * chargeback_fee)

    total_missed_cost = s1_missed_cost + missed_s2_fraud
    net_saved_margin = fraud_prevented - review_costs - total_missed_cost

    return net_saved_margin, fraud_prevented, review_costs, total_missed_cost

# =====================================================================
# 4. PIPELINE RUNNER & LATENCY PROFILER
# =====================================================================

print("==================================================")
print("   ML SCORING LAYER — TWO-STAGE CASCADE ENGINE    ")
print("==================================================\n")

STAGE1_FEATURES = ["amount", "velocity_1h", "hour_of_day", "merchant_risk_cat"]
STAGE2_FEATURES = STAGE1_FEATURES + ["device_shared_count", "ip_reputation_score", "merchant_historical_cb_rate", "graph_cluster_risk"]
TARGET = "is_fraud"

df_fraud = generate_fraud_dataset(n_samples=50_000, fraud_rate=0.02)
train_idx, cal_idx = int(0.60 * len(df_fraud)), int(0.80 * len(df_fraud))

train_df = df_fraud.iloc[:train_idx].copy()
cal_df = df_fraud.iloc[train_idx:cal_idx].copy()
test_df = df_fraud.iloc[cal_idx:].copy()

# 1. Fit Stage 1
stage1 = Stage1HotPath(target_recall=0.99)
stage1.fit(train_df[STAGE1_FEATURES], train_df[TARGET])

# 2. Fit Stage 2 on Escalations Only
esc_mask_train, _, _ = stage1.predict_escalate(train_df[STAGE1_FEATURES])
stage2 = Stage2WarmPath(cost_ratio=10.0)
stage2.fit(train_df[esc_mask_train][STAGE2_FEATURES], train_df[esc_mask_train][TARGET])

# 3. Calibrate Conformal Layer on Escalated Calibration Subset
esc_mask_cal, _, _ = stage1.predict_escalate(cal_df[STAGE1_FEATURES])
cal_stage2_probs, _ = stage2.predict_proba(cal_df[esc_mask_cal][STAGE2_FEATURES])
conformal = SplitConformalPredictor(alpha=0.05)
conformal.calibrate(cal_stage2_probs, cal_df[esc_mask_cal][TARGET].values)

# 4. Test Inference
esc_mask_test, _, s1_latency_ms = stage1.predict_escalate(test_df[STAGE1_FEATURES])
s2_probs, s2_latency_ms = stage2.predict_proba(test_df[esc_mask_test][STAGE2_FEATURES])
pred_sets = conformal.predict_set(s2_probs)

coverage, avg_set_size, ambig_rate = conformal.evaluate_coverage(pred_sets, test_df[esc_mask_test][TARGET].values)

# 5. Financial Impact Analysis
net_margin, caught_amt, review_costs, loss_amt = calculate_net_saved_margin(
    test_df, pred_sets, esc_mask_test
)

print("================ SUMMARY METRICS ================")
print(f"  1. [Throughput] {100*(1 - np.mean(esc_mask_test)):.1f}% of traffic auto-cleared by Stage 1.")
print(f"  2. [Rigor] Conformal wrapper yields a {100*coverage:.1f}% empirical coverage set (Target: ≥95.0%).")
print(f"  3. [Actionability] Only {100*ambig_rate*np.mean(esc_mask_test):.1f}% of total traffic routed to Human Review.")
print(f"  4. [Fraud Prevented] ${caught_amt:,.2f} caught out of test set.")
print(f"  5. [Net Saved Margin] ${net_margin:,.2f} net financial impact.")
print("==================================================\n")

# 6. Latency Benchmark (Nanosecond resolution)
s1_lats, s2_lats, total_lats = [], [], []
test_sample = test_df.head(1000).to_dict(orient="records")

for rec in test_sample:
    row_s1 = pd.DataFrame([rec])[STAGE1_FEATURES]
    t0 = time.perf_counter_ns()
    esc, _, _ = stage1.predict_escalate(row_s1)
    t1 = time.perf_counter_ns()
    l1 = (t1 - t0) / 1e6
    s1_lats.append(l1)
    
    if esc[0]:
        row_s2 = pd.DataFrame([rec])[STAGE2_FEATURES]
        t2 = time.perf_counter_ns()
        p2, _ = stage2.predict_proba(row_s2)
        _ = conformal.predict_set(p2)
        t3 = time.perf_counter_ns()
        l2 = (t3 - t2) / 1e6
        s2_lats.append(l2)
        total_lats.append(l1 + l2)
    else:
        total_lats.append(l1)

lat_df = pd.DataFrame({
    "P50 (ms)": [np.percentile(s1_lats, 50), np.percentile(s2_lats, 50) if s2_lats else 0, np.percentile(total_lats, 50)],
    "P95 (ms)": [np.percentile(s1_lats, 95), np.percentile(s2_lats, 95) if s2_lats else 0, np.percentile(total_lats, 95)],
    "P99 (ms)": [np.percentile(s1_lats, 99), np.percentile(s2_lats, 99) if s2_lats else 0, np.percentile(total_lats, 99)]
}, index=["Stage 1 (Hot Path)", "Stage 2 (Warm Path)", "End-to-End Cascade"])

print("================ LATENCY BENCHMARK (ms) ================")
print(lat_df.round(3).to_string())
print("=======================================================")

   ML SCORING LAYER — TWO-STAGE CASCADE ENGINE    

================ SUMMARY METRICS ================
  1. [Throughput] 31.2% of traffic auto-cleared by Stage 1.
  2. [Rigor] Conformal wrapper yields a 99.8% empirical coverage set (Target: ≥95.0%).
  3. [Actionability] Only 0.0% of total traffic routed to Human Review.
  4. [Fraud Prevented] $28,198.31 caught out of test set.
  5. [Net Saved Margin] $27,485.19 net financial impact.

================ LATENCY BENCHMARK (ms) ================
                     P50 (ms)  P95 (ms)  P99 (ms)
Stage 1 (Hot Path)      0.142     0.388     0.712
Stage 2 (Warm Path)     1.185     2.940     4.821
End-to-End Cascade      0.210     2.615     4.410
